In [34]:
%%writefile test.cu
#include <stdio.h>
#include <time.h>

#define BLOCK_SIZE  16          // submatrix size

__global__ void matMult ( float * a, float * b, int m, int n, int k, float * c )
{
    int   bx  = blockIdx.x;     // block index (по K)
    int   by  = blockIdx.y;     // block index (по M)
    int   tx  = threadIdx.x;    // thread index (по K внутри блока)
    int   ty  = threadIdx.y;    // thread index (по M внутри блока)
    float sum = 0.0f;           // computed subelement

    // Глобальные индексы
    int row = by * BLOCK_SIZE + ty;  // строка в матрице C (0..M-1)
    int col = bx * BLOCK_SIZE + tx;  // столбец в матрице C (0..K-1)

    // Проверка границ
    if (row < m && col < k) {
        // Умножение row-й строки A на col-й столбец B
        for (int i = 0; i < n; i++) {
            sum += a[row * n + i] * b[i * k + col];
        }

        // Запись результата
        c[row * k + col] = sum;
    }
}

void matMultCPUSimple(float* A, float* B, float* C, int M, int N, int K) {
    // Перебираем все элементы результирующей матрицы
    for (int row = 0; row < M; row++) {
        for (int col = 0; col < K; col++) {
            float sum = 0;
            // Скалярное произведение строки A и столбца B
            for (int i = 0; i < N; i++) {
                sum += A[row * N + i] * B[i * K + col];
            }
            C[row * K + col] = sum;
        }
    }
    printf("\nFirst few elements of C (CPU result):\n");
    for (int i = 0; i < (5 < M ? 5 : M); i++) {
        for (int j = 0; j < (5 < K ? 5 : K); j++) {
            printf("%8.2f ", C[i * K + j]);
        }
        printf("\n");
    }
}

int main ( int argc, char *  argv [] )
{
    // Размеры матриц: A (M x N), B (N x K), C (M x K)
    int M = 1024;  // количество строк в A и C
    int N = 512;   // количество столбцов в A и строк в B
    int K = 768;   // количество столбцов в B и C

    int numBytesA = M * N * sizeof ( float );
    int numBytesB = N * K * sizeof ( float );
    int numBytesC = M * K * sizeof ( float );

    // allocate host memory
    float * a = new float [M * N];
    float * b = new float [N * K];
    float * c = new float [M * K];
    float * c_cpu = new float [M * K];  // отдельный массив для CPU результата

    // Инициализация матриц
    for ( int i = 0; i < M; i++ )
        for ( int j = 0; j < N; j++ )
            a[i * N + j] = (i == j) ? 1.0f : 0.0f;  // единичная матрица

    for ( int i = 0; i < N; i++ )
        for ( int j = 0; j < K; j++ )
            b[i * K + j] = 2.0f;  // матрица из двоек

    // CPU вычисление
    clock_t start_cpu, end_cpu;
    start_cpu = clock();
    matMultCPUSimple(a, b, c_cpu, M, N, K);
    end_cpu = clock();
    printf("CPU Simple version: %.2f ms\n", (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC * 1000);

    // allocate device memory
    float * adev = NULL;
    float * bdev = NULL;
    float * cdev = NULL;

    cudaMalloc ( (void**)&adev, numBytesA );
    cudaMalloc ( (void**)&bdev, numBytesB );
    cudaMalloc ( (void**)&cdev, numBytesC );

    // set kernel launch configuration
    dim3 threads ( BLOCK_SIZE, BLOCK_SIZE );
    dim3 blocks  ( (K + BLOCK_SIZE - 1) / BLOCK_SIZE,
                   (M + BLOCK_SIZE - 1) / BLOCK_SIZE );

    // create cuda event handles
    cudaEvent_t start_gpu, stop_gpu;
    float gpuTime = 0.0f;

    cudaEventCreate ( &start_gpu );
    cudaEventCreate ( &stop_gpu );

    // asynchronously issue work to the GPU (all to stream 0)
    cudaEventRecord ( start_gpu, 0 );
    cudaMemcpy      ( adev, a, numBytesA, cudaMemcpyHostToDevice );
    cudaMemcpy      ( bdev, b, numBytesB, cudaMemcpyHostToDevice );

    matMult<<<blocks, threads>>> ( adev, bdev, M, N, K, cdev );

    cudaMemcpy      ( c, cdev, numBytesC, cudaMemcpyDeviceToHost );
    cudaEventRecord ( stop_gpu, 0 );

    cudaEventSynchronize ( stop_gpu );
    cudaEventElapsedTime ( &gpuTime, start_gpu, stop_gpu );

    // print the gpu time
    printf("\nMatrix sizes: A(%d x %d), B(%d x %d), C(%d x %d)\n", M, N, N, K, M, K);
    printf("GPU time: %.2f milliseconds\n", gpuTime );

    // Проверка результата
    printf("\nFirst few elements of C (GPU result):\n");
    for (int i = 0; i < (5 < M ? 5 : M); i++) {
        for (int j = 0; j < (5 < K ? 5 : K); j++) {
            printf("%8.2f ", c[i * K + j]);
        }
        printf("\n");
    }

    // Верификация результатов
    float maxDiff = 0.0f;
    for (int i = 0; i < M * K; i++) {
        float diff = c_cpu[i] - c[i];
        if (diff < 0) diff = -diff;
        if (diff > maxDiff) maxDiff = diff;
    }


    // release resources
    cudaEventDestroy ( start_gpu );
    cudaEventDestroy ( stop_gpu  );
    cudaFree         ( adev  );
    cudaFree         ( bdev  );
    cudaFree         ( cdev  );

    delete[] a;
    delete[] b;
    delete[] c;
    delete[] c_cpu;

    return 0;
}

Overwriting test.cu


In [35]:
!nvcc test.cu -o test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [36]:
!./test


First few elements of C (CPU result):
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
CPU Simple version: 1643.00 ms

Matrix sizes: A(1024 x 512), B(512 x 768), C(1024 x 768)
GPU time: 6.16 milliseconds

First few elements of C (GPU result):
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
    2.00     2.00     2.00     2.00     2.00 
